In [ ]:
import numpy as np
from math import log
from tqdm import tqdm
 
from Welsch import AlphaDivergenceAlgo
from Huber import HuberAlgo
from help_functions import generate_linear_model, grid_search_cv_tukey,grid_search_cv_hampel
import statsmodels.api as sm
from statsmodels.robust.norms import TukeyBiweight
from sklearn.model_selection import KFold, train_test_split
from statsmodels.robust.robust_linear_model import RLM
from statsmodels.robust.norms import Hampel
import pandas as pd

# Break Down point

## General Setup for the break down point experiments

In [ ]:
N_SAMPLES = 10_000
P_FEATURES = 10
BETA_SCALE = 1_000
OUTLIER_SCALE =10000000
DELTA = 0.01
NUM_REPETITIONS = 5_000
EPSILON_RANGE = np.arange(0.0, 0.3, 0.01)
MAX_ITER = 100
SEED = 100
 
np.random.seed(SEED)
beta_star = BETA_SCALE * np.random.choice([-1, 1], size=P_FEATURES)
tau_base = log(1 / DELTA) / N_SAMPLES

## Monte Carlo Experminets

In [ ]:
break_down_Welsch= []
 
for epsilon in EPSILON_RANGE:
    tau = tau_base + epsilon
    squared_errors = []
    beta_hats = []
 
    for rep in tqdm(range(NUM_REPETITIONS), desc=f"epsilon={epsilon:.2f}"):
        y,X,_ = generate_linear_model(
            N_SAMPLES,
            P_FEATURES,
            beta_star,
            seed=rep,
            outliers=True,
            outlier_const=OUTLIER_SCALE,
            noise_type="pareto",
            outliers_perc=epsilon,
        )
 
        model = AlphaDivergenceAlgo(X, y)
        beta_hat = model.optimizer_approach(tau=tau, maxiter=MAX_ITER)
 
        beta_hats.append(beta_hat)
        squared_errors.append(np.linalg.norm(beta_star - beta_hat) ** 2)
 
    bias = np.linalg.norm(beta_star - np.mean(beta_hats, axis=0))
    rmse = np.sqrt(np.mean(squared_errors))
    break_down_Welsch.append((epsilon, bias, rmse))

In [ ]:
break_down_huber = []
 
for epsilon in EPSILON_RANGE:
    squared_errors = []
    beta_hats = []
    
 
    for rep in tqdm(range(NUM_REPETITIONS), desc=f"epsilon={epsilon:.2f}"):
        gamma = log(1 / DELTA) / N_SAMPLES + epsilon
        y,X,_ = generate_linear_model(
            N_SAMPLES,
            P_FEATURES,
            beta_star,
            seed=rep,
            outliers=True,
            outlier_const=OUTLIER_SCALE,
            noise_type="pareto",
            outliers_perc=epsilon,
        )
        model = HuberAlgo(X, y)
        beta_hat, _ = model.optimizer_approach(gamma=1, max_iter=MAX_ITER)
 
        beta_hats.append(beta_hat)
        squared_errors.append(np.linalg.norm(beta_star - beta_hat) ** 2)
 
    bias = np.linalg.norm(beta_star - np.mean(beta_hats, axis=0))
    rmse = np.sqrt(np.mean(squared_errors))
    break_down_huber.append((epsilon, bias, rmse))


In [ ]:
break_down_tukey = []
c_values=np.arange(0.5, 5.5, 0.5)
 
for epsilon in EPSILON_RANGE:
    print(f"Running epsilon={epsilon:.2f}")
    squared_errors = []
    beta_hats = []
 
    for rep in tqdm(range(NUM_REPETITIONS), desc=f"epsilon={epsilon:.2f}"):
        y, X, _ = generate_linear_model(
            N_SAMPLES,
            P_FEATURES,
            beta_star,
            seed=rep,
            outliers=True,
            outlier_const=OUTLIER_SCALE,
            noise_type="pareto",
            outliers_perc=epsilon,
        )
 
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=0.1, shuffle=True, random_state=rep
        )
 
        # Tune Tukey constant on validation set
        best_c = grid_search_cv_tukey(
            X_val, y_val, c_values,
            n_splits=5
        )
 
        # Refit on training set with best parameters
        rlm_model = sm.RLM(y_train, X_train, M=TukeyBiweight(c=best_c))
        beta_hat = rlm_model.fit().params
 
        beta_hats.append(beta_hat)
        squared_errors.append(np.linalg.norm(beta_star - beta_hat) ** 2)
 
    bias = np.linalg.norm(beta_star - np.mean(beta_hats, axis=0))
    rmse = np.sqrt(np.mean(squared_errors))
    break_down_tukey.append((epsilon, bias, rmse, best_c))
 

In [ ]:
break_down_hampel = []
A_VALUES = [1.5, 2.0, 2.5,3,3.5,4.0]
B_VALUES = [3.0, 4.0, 5.0,6.0,7.0,8.0]
C_VALUES = [6.0, 8.0, 10.0,12.0,14.0,15.0]
 
for epsilon in EPSILON_RANGE:
    print(f"Running epsilon={epsilon:.2f}")
    squared_errors = []
    beta_hats = []
 
    for rep in tqdm(range(NUM_REPETITIONS), desc=f"epsilon={epsilon:.2f}"):
        y, X, _ = generate_linear_model(
            N_SAMPLES,
            P_FEATURES,
            beta_star,
            seed=rep,
            outliers=True,
            outlier_const=OUTLIER_SCALE,
            noise_type="pareto",
            outliers_perc=epsilon,
        )
 
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=0.1, shuffle=True, random_state=rep
        )
 
        # Tune Hampel constants on validation set
        best_params = grid_search_cv_hampel(
            X_val, y_val, A_VALUES, B_VALUES, C_VALUES,
            n_splits=5, random_state=rep,
        )
 
        # Refit on training set with best parameters
        model = RLM(y_train, X_train, M=Hampel(**best_params))
        beta_hat = model.fit().params
 
        beta_hats.append(beta_hat)
        squared_errors.append(np.linalg.norm(beta_star - beta_hat) ** 2)
 
    bias = np.linalg.norm(beta_star - np.mean(beta_hats, axis=0))
    rmse = np.sqrt(np.mean(squared_errors))
    break_down_hampel.append((epsilon, bias, rmse))

In [ ]:
break_down_quantile_1 = []
 
for epsilon in EPSILON_RANGE:
    print(f"Running epsilon={epsilon:.2f}")
    squared_errors = []
    beta_hats = []
 
    for rep in tqdm(range(NUM_REPETITIONS), desc=f"epsilon={epsilon:.2f}"):
        y, X, _ = generate_linear_model(
            N_SAMPLES,
            P_FEATURES,
            beta_star,
            seed=rep,
            outliers=True,
            outlier_const=OUTLIER_SCALE,
            noise_type="pareto",
            outliers_perc=epsilon,
        )
 
        model = sm.QuantReg(y, X)
        beta_hat = model.fit(q=0.1).params
 
        beta_hats.append(beta_hat)
        squared_errors.append(np.linalg.norm(beta_star - beta_hat) ** 2)
 
    bias = np.linalg.norm(beta_star - np.mean(beta_hats, axis=0))
    rmse = np.sqrt(np.mean(squared_errors))
    break_down_quantile_1.append((epsilon, bias, rmse))

In [ ]:
break_down_quantile_5 = []
 
for epsilon in EPSILON_RANGE:
    print(f"Running epsilon={epsilon:.2f}")
    squared_errors = []
    beta_hats = []
 
    for rep in tqdm(range(NUM_REPETITIONS), desc=f"epsilon={epsilon:.2f}"):
        y, X, _ = generate_linear_model(
            N_SAMPLES,
            P_FEATURES,
            beta_star,
            seed=rep,
            outliers=True,
            outlier_const=OUTLIER_SCALE,
            noise_type="pareto",
            outliers_perc=epsilon,
        )
 
        model = sm.QuantReg(y, X)
        beta_hat = model.fit(q=0.5).params
 
        beta_hats.append(beta_hat)
        squared_errors.append(np.linalg.norm(beta_star - beta_hat) ** 2)
 
    bias = np.linalg.norm(beta_star - np.mean(beta_hats, axis=0))
    rmse = np.sqrt(np.mean(squared_errors))
    break_down_quantile_5.append((epsilon, bias, rmse))

In [ ]:
break_down_quantile_9 = []
 
for epsilon in EPSILON_RANGE:
    print(f"Running epsilon={epsilon:.2f}")
    squared_errors = []
    beta_hats = []
 
    for rep in tqdm(range(NUM_REPETITIONS), desc=f"epsilon={epsilon:.2f}"):
        y, X, _ = generate_linear_model(
            N_SAMPLES,
            P_FEATURES,
            beta_star,
            seed=rep,
            outliers=True,
            outlier_const=OUTLIER_SCALE,
            noise_type="pareto",
            outliers_perc=epsilon,
        )
 
        model = sm.QuantReg(y, X)
        beta_hat = model.fit(q=0.9).params
 
        beta_hats.append(beta_hat)
        squared_errors.append(np.linalg.norm(beta_star - beta_hat) ** 2)
 
    bias = np.linalg.norm(beta_star - np.mean(beta_hats, axis=0))
    rmse = np.sqrt(np.mean(squared_errors))
    break_down_quantile_5.append((epsilon, bias, rmse))

## Plots

Since the above experiments are long we provide the results we got in the folder results_break_down_points

In [ ]:
break_Welsch= pd.read_csv('./break_down_points/break_down_exp_Welsch.csv')

In [ ]:
break_down_Welsch_df = pd.DataFrame(break_Welsch, columns=['delta', 'Error', 'std','itterations'])

In [ ]:
break_down_huber=pd.read_csv('./break_down_points/break_down_exp_Hube.csv')

In [ ]:
break_down_huber_df = pd.DataFrame(break_down_huber, columns=['delta', 'Error', 'std','itterations'])

In [ ]:
break_down_tukey=pd.read_csv('./break_down_points/break_down_exp_tukey.csv')

In [ ]:
break_down_tukey_df = pd.DataFrame(break_down_tukey, columns=['delta', 'Error', 'std','itterations'])

In [ ]:
break_down_hampel=pd.read_csv('./break_down_points/break_down_exp_Hampel.csv')

In [ ]:
break_down_hampel_df = pd.DataFrame(break_down_hampel, columns=['delta', 'Error'])